## EXPERIMENT 4: Image Classification using Neural Network with R + Keras 3

### 1. Install Packages

In [1]:
install.packages(c("jpeg", "keras3"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘RcppTOML’, ‘here’, ‘png’, ‘config’, ‘tfautograph’, ‘reticulate’, ‘tensorflow’, ‘tfruns’, ‘zeallot’, ‘dotty’




### 2. Load Packages

In [2]:
library(jpeg)
library(keras3)

### 3. Read Images

In [3]:
# In Google Colab, upload your images to:
# /content/images

setwd("/content/sample_data/images")

pics <- c(
  "p1.jpg", "p2.jpg", "p3.jpg",
  "p4.jpg", "p5.jpg", "p6.jpg",
  "c1.jpg", "c2.jpg", "c3.jpg",
  "c4.jpg", "c5.jpg", "c6.jpg"
)

mypic <- list()

for (i in 1:12) {
  mypic[[i]] <- readJPEG(pics[i])
}

ERROR: Error in setwd("/content/sample_data/images"): cannot change working directory


### 4. Explore Images

In [ ]:
print(mypic[[1]])

# Display image
plot(
  as.raster(mypic[[8]]),
  main = "Image 8"
)

# Summary
summary(mypic[[1]])

# Histogram
hist(
  as.numeric(mypic[[2]]),
  main = "Pixel Histogram",
  xlab = "Pixel Value"
)

# Structure
str(mypic)

### 5. Resize Images to 28 x 28

In [ ]:
# Simple nearest-neighbor resize using base R

resize_image <- function(img, width = 28, height = 28) {

  old_height <- dim(img)[1]
  old_width  <- dim(img)[2]

  row_index <- round(
    seq(1, old_height, length.out = height)
  )

  col_index <- round(
    seq(1, old_width, length.out = width)
  )

  row_index[row_index < 1] <- 1
  col_index[col_index < 1] <- 1

  resized <- img[row_index, col_index, , drop = FALSE]

  return(resized)
}


for (i in 1:12) {

  mypic[[i]] <- resize_image(
    mypic[[i]],
    28,
    28
  )

}


# Check dimensions

for (i in 1:12) {
  print(dim(mypic[[i]]))
}

### 6. Convert Images to Vectors

In [ ]:
# 28 x 28 x 3 = 2352 features

image_to_vector <- function(img) {
  as.numeric(img)
}

### 7. Create Training Data

In [ ]:
# Original experiment:
# images 7-11 = training
# images 6 and 12 = testing

trainx <- NULL

for (i in 7:11) {
  trainx <- rbind(
    trainx,
    image_to_vector(mypic[[i]])
  )
}

testx <- rbind(
  image_to_vector(mypic[[6]]),
  image_to_vector(mypic[[12]])
)

### 8. Create Labels

In [ ]:
trainy <- c(
  0, 0, 0, 0, 0,
  1, 1, 1, 1, 1
)

testy <- c(
  0, 1
)

### Check Dimensions

In [ ]:
print(dim(trainx))
print(dim(testx))

print(trainy)
print(testy)

### 9. One-Hot Encoding

In [ ]:
# Avoid to_categorical()
# Use base R instead

trainLabels <- cbind(
  as.numeric(trainy == 0),
  as.numeric(trainy == 1)
)

testLabels <- cbind(
  as.numeric(testy == 0),
  as.numeric(testy == 1)
)

print(trainLabels)
print(testLabels)

### 10. Build Neural Network

In [ ]:
model <- keras_model_sequential(
  input_shape = c(2352)
) |>
  layer_dense(
    units = 256,
    activation = "relu"
  ) |>
  layer_dense(
    units = 128,
    activation = "relu"
  ) |>
  layer_dense(
    units = 2,
    activation = "softmax"
  )


# Display model

summary(model)

### 11. Compile Model

In [ ]:
compile(
  model,

  optimizer = optimizer_adam(),

  loss = "categorical_crossentropy",

  metrics = c("accuracy")
)

### 12. Train Model

In [ ]:
history <- fit(
  model,

  trainx,

  trainLabels,

  epochs = 30,

  batch_size = 2,

  validation_split = 0.2
)

### 13. Evaluate Model

In [ ]:
results <- evaluate(
  model,

  testx,

  testLabels
)

print(results)

### 14. Predict

In [ ]:
prob <- predict(
  model,
  testx
)

print(prob)

### 15. Convert Probability to Class

In [ ]:
pred <- max.col(prob) - 1

print(pred)

### 16. Confusion Matrix

In [ ]:
table(
  Predicted = pred,
  Actual = testy
)

### 17. Final Results

In [ ]:
results_table <- cbind(
  prob,
  Predicted = pred,
  Actual = testy
)

print(results_table)